<a href="https://colab.research.google.com/github/shreyansh8h/hello-world/blob/master/Copy_of_3_Prompt_testing_openi_fixed_Mistral_Planner_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prompt Testing with Hugging Face (Mistral)

This notebook is cleaned up for reliability and beginner readability.

What was fixed:
- removed OpenAI key dependency
- deterministic model settings
- robust JSON parsing
- reusable helper functions
- better error handling


In [1]:
# If needed in Colab/Jupyter, uncomment:
!pip -q install -U "langchain-huggingface" "transformers>=4.45,<4.58" "tokenizers<0.22" accelerate bitsandbytes


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
import re
import json
from typing import Any, Dict

import torch
from transformers import BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace


In [ ]:
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

use_4bit = torch.cuda.is_available()
quantization_config = None
if use_4bit:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

model_kwargs = {"device_map": "auto"}
if quantization_config is not None:
    model_kwargs["quantization_config"] = quantization_config

hf_llm = HuggingFacePipeline.from_model_id(
    model_id=MODEL_ID,
    task="text-generation",
    pipeline_kwargs={
        "max_new_tokens": 800,
        "do_sample": False,
        "return_full_text": False,
    },
    model_kwargs=model_kwargs,
)

llm = ChatHuggingFace(llm=hf_llm)
print(f"Model ready: {MODEL_ID} | 4-bit: {use_4bit}")


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Model ready: mistralai/Mistral-7B-Instruct-v0.3 | 4-bit: True


In [ ]:
def parse_json_content(content: str) -> Dict[str, Any]:
    """Parse model text into JSON. Handles fenced JSON blocks too."""
    text = content.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\\s*", "", text)
        text = re.sub(r"\\s*```$", "", text)
    return json.loads(text)


def run_text_task(title: str, prompt: str) -> str:
    """Run a non-JSON prompt and print result with clear labeling."""
    try:
        response = llm.invoke(prompt)
        result = response.content.strip()
        print(f"{title}:\\n{result}\\n")
        return result
    except Exception as e:
        print(f"{title} failed: {e}")
        return ""


def run_json_task(title: str, prompt: str) -> Dict[str, Any]:
    """Run a JSON extraction prompt with robust parsing."""
    try:
        response = llm.invoke(prompt + "\n\nReturn valid JSON only. No markdown.")
        parsed = parse_json_content(response.content)
        print(f"{title}:")
        print(json.dumps(parsed, indent=2))
        print()
        return parsed
    except Exception as e:
        print(f"{title} failed: {e}")
        return {}


In [ ]:
# Sample review text
review_text = """
Product: SmartHeadphones X1
Rating: 4.5/5
Reviewer: Jane Doe
Date: June 15, 2025
Comment: The SmartHeadphones X1 deliver crisp sound and excellent noise cancellation.
Battery life is impressive at 20 hours, but the app could be more user-friendly.
Price: $149.99
""".strip()

# 1) Zero-shot extraction (JSON)
prompt = f"""
Extract key-value pairs from the product review below.
Return valid JSON only.

Text:
{review_text}
"""
run_json_task("1) Zero-Shot Extraction", prompt)


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


1) Zero-Shot Extraction:
{
  "Product": "SmartHeadphones X1",
  "Rating": "4.5/5",
  "Reviewer": "Jane Doe",
  "Date": "June 15, 2025",
  "Comment": "The SmartHeadphones X1 deliver crisp sound and excellent noise cancellation. Battery life is impressive at 20 hours, but the app could be more user-friendly.",
  "Price": "$149.99"
}



{'Product': 'SmartHeadphones X1',
 'Rating': '4.5/5',
 'Reviewer': 'Jane Doe',
 'Date': 'June 15, 2025',
 'Comment': 'The SmartHeadphones X1 deliver crisp sound and excellent noise cancellation. Battery life is impressive at 20 hours, but the app could be more user-friendly.',
 'Price': '$149.99'}

In [ ]:
# 2) Few-shot extraction (JSON)
prompt = f"""
Extract key-value pairs from the product review below.
Return valid JSON only.

Example input:
Product: Wireless Mouse
Rating: 4/5
Reviewer: John Smith
Date: May 1, 2024
Price: $29.99

Example output:
{{
  "Product": "Wireless Mouse",
  "Rating": "4/5",
  "Reviewer": "John Smith",
  "Date": "May 1, 2024",
  "Price": "$29.99"
}}

Text:
{review_text}
"""
run_json_task("2) Few-Shot Extraction", prompt)


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


2) Few-Shot Extraction:
{
  "Product": "SmartHeadphones X1",
  "Rating": "4.5/5",
  "Reviewer": "Jane Doe",
  "Date": "June 15, 2025",
  "Comment": "The SmartHeadphones X1 deliver crisp sound and excellent noise cancellation. Battery life is impressive at 20 hours, but the app could be more user-friendly.",
  "Price": "$149.99"
}



{'Product': 'SmartHeadphones X1',
 'Rating': '4.5/5',
 'Reviewer': 'Jane Doe',
 'Date': 'June 15, 2025',
 'Comment': 'The SmartHeadphones X1 deliver crisp sound and excellent noise cancellation. Battery life is impressive at 20 hours, but the app could be more user-friendly.',
 'Price': '$149.99'}

In [ ]:
# 3) Chain-of-thought style question (final answer only)
prompt = f"""
Given this product review, identify the product strengths.
Think step by step. Keep the thinking crisp and clear

Text:
{review_text}
"""
run_text_task("3) Product Strengths", prompt)

# 4) Role-based
prompt = f"""
You are a product analyst.
Classify sentiment as Positive, Negative, or Mixed with one-line reason.
Use only the comment text.

Text:
{review_text}
"""
run_text_task("4) Role-Based Sentiment", prompt)

# 5) Instruction-based
prompt = f"""
Answer in one sentence using exact values from the text.
Question: What is the battery life?

Text:
{review_text}
"""
run_text_task("5) Instruction-Based QA", prompt)


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


3) Product Strengths:\nThe product strengths of the SmartHeadphones X1, as identified from the review, are:

1. Sound Quality: The reviewer mentions that the SmartHeadphones X1 delivers crisp sound, indicating that the audio quality is clear and detailed.

2. Noise Cancellation: The reviewer praises the headphones for their excellent noise cancellation, suggesting that they effectively reduce background noise for a more immersive listening experience.

3. Battery Life: The battery life of 20 hours is impressive, meaning the headphones can last for extended periods without needing to be recharged, making them convenient for travel or long listening sessions.

4. Price: The headphones are priced at $149.99, which is competitive in the market for high-quality headphones. This price point may appeal to consumers looking for a balance between quality and affordability.

However, it's worth noting that the reviewer also mentions a potential weakness in the user-friendly nature of the app ass

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


4) Role-Based Sentiment:\nPositive: The SmartHeadphones X1 deliver crisp sound, excellent noise cancellation, and an impressive battery life of 20 hours, but the app could be more user-friendly.\n
5) Instruction-Based QA:\nThe battery life of the SmartHeadphones X1 is 20 hours.\n


'The battery life of the SmartHeadphones X1 is 20 hours.'

In [ ]:
# Order text sample
order_text = """
Order Confirmation
Order Number: ORD-2025-789
Customer Name: Sarah Johnson
Email: sarah.johnson@email.com
Order Date: July 20, 2025
Delivery Address: 123 Maple Street, Springfield, IL 62701
Items Ordered:
- Laptop Pro 14" (1 unit, $1299.99)
- Wireless Mouse (2 units, $29.99 each)
Total Amount: $1359.97
Payment Method: Credit Card
Estimated Delivery: July 25, 2025
""".strip()

prompt = f"""
Extract all entities and key-value pairs from this order.
Return valid JSON only.

Text:
{order_text}
"""
run_json_task("6) Order Extraction", prompt)


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


6) Order Extraction:
{
  "Order Number": "ORD-2025-789",
  "Customer Name": "Sarah Johnson",
  "Email": "sarah.johnson@email.com",
  "Order Date": "July 20, 2025",
  "Delivery Address": {
    "street": "123 Maple Street",
    "city": "Springfield",
    "state": "IL",
    "zip_code": "62701"
  },
  "Items Ordered": [
    {
      "product_name": "Laptop Pro 14\"",
      "quantity": 1,
      "price": 1299.99
    },
    {
      "product_name": "Wireless Mouse",
      "quantity": 2,
      "price": 29.99
    }
  ],
  "Total Amount": 1359.97,
  "Payment Method": "Credit Card",
  "Estimated Delivery": "July 25, 2025"
}



{'Order Number': 'ORD-2025-789',
 'Customer Name': 'Sarah Johnson',
 'Email': 'sarah.johnson@email.com',
 'Order Date': 'July 20, 2025',
 'Delivery Address': {'street': '123 Maple Street',
  'city': 'Springfield',
  'state': 'IL',
  'zip_code': '62701'},
 'Items Ordered': [{'product_name': 'Laptop Pro 14"',
   'quantity': 1,
   'price': 1299.99},
  {'product_name': 'Wireless Mouse', 'quantity': 2, 'price': 29.99}],
 'Total Amount': 1359.97,
 'Payment Method': 'Credit Card',
 'Estimated Delivery': 'July 25, 2025'}

In [ ]:
# Long legal sample (truncated to keep demo fast and reduce token cost)
sample_text = """
COMMERCIAL LOAN AGREEMENT
Dated as of March 2, 2022
Between 11 Waterview Blvd. LLC, a New Jersey limited liability company, as Borrower
and RED OAK CAPITAL FUND V, LLC, a Delaware limited liability company, as Lender.

"Affiliate" means, as to any Person, any other Person that, directly or indirectly, is in Control of,
is Controlled by, or is under common Control with, such Person.

"Annual Budget" means an operating budget, including Borrower's good faith estimate of anticipated
Gross Income from Operations, Operating Expenses and planned Capital Expenditures.

"Availability Threshold" means the greater of $250,000 or 1% of the initial principal balance of the Loan.
""".strip()

prompt = f"""
You are an entity extraction expert.
Extract structured fields from this legal text as valid JSON only.
Use this schema:
{{
  "document_type": "",
  "date": "",
  "parties": [{{"name": "", "role": "", "jurisdiction": ""}}],
  "definitions": [{{"term": "", "definition": ""}}],
  "numeric_terms": [{{"name": "", "value": ""}}]
}}

Text:
{sample_text}
"""
run_json_task("7) Legal Text Extraction", prompt)


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


7) Legal Text Extraction:
{
  "document_type": "Commercial Loan Agreement",
  "date": "March 2, 2022",
  "parties": [
    {
      "name": "11 Waterview Blvd. LLC",
      "role": "Borrower",
      "jurisdiction": "New Jersey"
    },
    {
      "name": "RED OAK CAPITAL FUND V, LLC",
      "role": "Lender",
      "jurisdiction": "Delaware"
    }
  ],
  "definitions": [
    {
      "term": "Affiliate",
      "definition": "As to any Person, any other Person that, directly or indirectly, is in Control of, is Controlled by, or is under common Control with, such Person."
    },
    {
      "term": "Annual Budget",
      "definition": "an operating budget, including Borrower's good faith estimate of anticipated Gross Income from Operations, Operating Expenses and planned Capital Expenditures."
    },
    {
      "term": "Availability Threshold",
      "definition": "the greater of $250,000 or 1% of the initial principal balance of the Loan."
    }
  ],
  "numeric_terms": [
    {
      "name":

{'document_type': 'Commercial Loan Agreement',
 'date': 'March 2, 2022',
 'parties': [{'name': '11 Waterview Blvd. LLC',
   'role': 'Borrower',
   'jurisdiction': 'New Jersey'},
  {'name': 'RED OAK CAPITAL FUND V, LLC',
   'role': 'Lender',
   'jurisdiction': 'Delaware'}],
 'definitions': [{'term': 'Affiliate',
   'definition': 'As to any Person, any other Person that, directly or indirectly, is in Control of, is Controlled by, or is under common Control with, such Person.'},
  {'term': 'Annual Budget',
   'definition': "an operating budget, including Borrower's good faith estimate of anticipated Gross Income from Operations, Operating Expenses and planned Capital Expenditures."},
  {'term': 'Availability Threshold',
   'definition': 'the greater of $250,000 or 1% of the initial principal balance of the Loan.'}],
 'numeric_terms': [{'name': 'Availability Threshold',
   'value': '$250,000 or 1% of the initial principal balance of the Loan'}]}

## Notes
- Keep `do_sample=False` for consistent classroom results.
- For extraction tasks, ask for JSON and parse carefully.
- For very long legal text, chunk input before extraction.
- In Colab without GPU, inference will be slower.


# Task Planner Agent


In [ ]:
# If needed:
!pip -q install -U langgraph langchain-community ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.7/173.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 122.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [ ]:
# Fix: don't import Tool from langchain.tools in your environment.
# Use DuckDuckGoSearchRun directly as the tool.

# If needed:
# !pip -q install -U langgraph langchain-community ddgs

from langgraph.prebuilt import create_react_agent
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()
tools = [search_tool]

system_prompt = """
You are an expert Task Planner.
Break down complex objectives into a clear, chronological, actionable plan.

Rules:
1) Use duckduckgo_search when facts or requirements need verification.
2) Keep steps practical and beginner-friendly.
3) Prioritize accuracy and avoid guessing.
4) Output sections:
   - Goal
   - Assumptions
   - 4-Week Plan (Week 1..4)
   - Daily Routine
   - Milestones
   - Risks + Mitigations
""".strip()

# llm = your existing ChatHuggingFace model object
planner_agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt,
)

result = planner_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Create a 4-week task plan for a student to learn Time Series forecasting with ARIMA and Prophet."
        }
    ]
})

print(result["messages"][-1].content)


/tmp/ipykernel_3190/57585583.py:31: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  planner_agent = create_react_agent(
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


 **Goal:**
To learn Time Series forecasting with ARIMA and Prophet models in Python.

**Assumptions:**
1. The student has a basic understanding of Python programming and statistical concepts.
2. The student has access to a computer with Python and necessary libraries installed.
3. The student is willing to dedicate at least 5 hours per week for learning.

**4-Week Plan:**

**Week 1:**
- Day 1:
  - Understand the basics of Time Series Analysis. (DuckDuckGo Search: "Time Series Analysis Basics")
  - Install necessary libraries: pandas, numpy, matplotlib, statsmodels, and fbprophet.

- Day 2:
  - Learn about ARIMA model, its components, and how to implement it in Python. (DuckDuckGo Search: "ARIMA Model Python")
  - Practice implementing ARIMA on a simple dataset. (DuckDuckGo Search: "ARIMA Python Examples")

- Day 3:
  - Learn about the Prophet library and its features. (DuckDuckGo Search: "Prophet Library Python")
  - Implement Prophet on a simple dataset. (DuckDuckGo Search: "Prophet P